# Lab 13: FID Score — Implementation & Experiments

**Goal:** Implement FID from scratch and use it to compare different generators.

What we'll do:
1. Build an Inception feature extractor
2. Extract features from real and fake images
3. Compute FID from scratch
4. Compare: random noise vs untrained GAN vs trained GAN
5. Track FID during training (watch it drop!)

**Expected compute** (approximate): Inception feature extraction dominates — 2,000 images ≈ 1–2 min on Apple-silicon (`mps`) or a Colab T4, ≈ 5–8 min on CPU. Whole notebook: ≈ 15 min on `mps`/T4, ≈ 60 min on CPU (CPU trains the DCGAN on a 10k-image subset). First run downloads the Inception v3 weights (~100 MB). The t-SNE experiment needs `scikit-learn`.

**Note:** this lab uses torchvision's Inception weights and 2,000 samples, so its FIDs are only comparable *within this notebook* — not to published numbers (those use the original TensorFlow weights via `pytorch-fid`, and 50,000 samples).

**How to use this lab:** each experiment has a **Predict** prompt — answer it before you run the cell.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from scipy.linalg import sqrtm
from torchvision import datasets, transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 1. Inception Feature Extractor

We hook into the **avgpool** layer (pool3) to grab 2048-dim features.

In [ ]:
class InceptionFeatureExtractor:
    def __init__(self, device):
        self.device = device
        self.model = inception_v3(weights=Inception_V3_Weights.DEFAULT)
        self.model.eval()
        self.model.to(device)

        # Hook to grab avgpool (pool3) output
        self.features = None
        self.model.avgpool.register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self.features = output.squeeze(-1).squeeze(-1)  # (batch, 2048)

    @torch.no_grad()
    def extract(self, images):
        """Extract 2048-dim features. Images must be preprocessed."""
        self.model(images.to(self.device))
        return self.features.cpu()

extractor = InceptionFeatureExtractor(device)
print("Inception feature extractor ready.")
print(f"Output: 2048-dim feature vector per image.")

## 2. Preprocessing

MNIST is 1x28x28 grayscale. Inception needs 3x299x299 with ImageNet normalization.

In [ ]:
def preprocess_for_inception(images):
    """
    Convert images from GAN output format to Inception input format.
    Input:  (batch, 1, 28, 28) in [-1, 1]
    Output: (batch, 3, 299, 299) with ImageNet normalization
    """
    # [-1, 1] -> [0, 1]
    images = (images + 1) / 2

    # 1 channel -> 3 channels
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)

    # Resize to 299x299
    images = torch.nn.functional.interpolate(
        images, size=(299, 299), mode='bilinear', align_corners=False
    )

    # ImageNet normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(images.device)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(images.device)
    images = (images - mean) / std

    return images

# Test it
test_img = torch.randn(2, 1, 28, 28)
processed = preprocess_for_inception(test_img)
print(f"Input:  {test_img.shape}")
print(f"Output: {processed.shape}")

## 3. FID Computation Functions

In [ ]:
def extract_all_features(images, extractor, batch_size=32):
    """Extract Inception features for a batch of images."""
    all_features = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        batch = preprocess_for_inception(batch)
        feats = extractor.extract(batch)
        all_features.append(feats)
    return torch.cat(all_features).numpy()


def compute_statistics(features):
    """Compute mean and covariance of feature vectors."""
    mu    = np.mean(features, axis=0)       # (2048,)
    sigma = np.cov(features, rowvar=False)   # (2048, 2048)
    return mu, sigma


def calculate_fid(mu_r, sigma_r, mu_f, sigma_f):
    """Compute Frechet Inception Distance."""
    # Term 1: squared distance between means
    diff = mu_r - mu_f
    mean_term = np.dot(diff, diff)

    # Term 2: trace of covariance comparison
    covmean = sqrtm(sigma_r @ sigma_f)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    trace_term = np.trace(sigma_r + sigma_f - 2 * covmean)

    return mean_term + trace_term


def compute_fid(real_images, fake_images, extractor, batch_size=32):
    """Full FID pipeline: images in, score out."""
    real_feats = extract_all_features(real_images, extractor, batch_size)
    fake_feats = extract_all_features(fake_images, extractor, batch_size)

    mu_r, sigma_r = compute_statistics(real_feats)
    mu_f, sigma_f = compute_statistics(fake_feats)

    return calculate_fid(mu_r, sigma_r, mu_f, sigma_f)

print("FID functions ready.")

## 4. Load Real Data & Prepare Generator

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
mnist = datasets.MNIST("data", train=True, download=True, transform=transform)
# CPU-only machines train the DCGAN on a 10k-image subset (None = all 60k images).
TRAIN_SUBSET = 10_000 if device.type == "cpu" else None
train_set = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET)) if TRAIN_SUBSET else mnist
loader = DataLoader(train_set, batch_size=64, shuffle=True)

# Collect real images (we'll use 2000 for speed — more is better)
n_eval = 2000
real_images = []
for img, _ in loader:
    real_images.append(img)
    if sum(x.size(0) for x in real_images) >= n_eval:
        break
real_images = torch.cat(real_images)[:n_eval]
print(f"Real images collected: {real_images.shape}")

In [ ]:
# DCGAN from Unit 1 (Build GANs), with z = 64 as in the Unit 2 labs
z_dim = 64

class Generator(nn.Module):
    def __init__(self, z_dim=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

print("Generator and Discriminator defined.")

## 5. Experiment 1: Baseline FIDs

Let's compute FID for different quality levels:
- **Random noise** (worst case)
- **Untrained Generator** (random weights)
- **Real vs Real** — two halves of the real set. Not exactly 0: 1,000 vs 1,000 images gives an upward small-sample bias (L13).

**Predict:** rank the three FIDs. Will real-vs-real be exactly 0?

In [ ]:
# Precompute real features (reuse for all comparisons)
print("Extracting real image features (this takes a moment)...")
real_feats = extract_all_features(real_images, extractor)
mu_r, sigma_r = compute_statistics(real_feats)
print(f"Real features shape: {real_feats.shape}")
print("Done!")

In [ ]:
# 1. Random noise
print("Computing FID for random noise...")
random_noise = torch.randn(n_eval, 1, 28, 28).clamp(-1, 1)
noise_feats = extract_all_features(random_noise, extractor)
mu_noise, sigma_noise = compute_statistics(noise_feats)
fid_noise = calculate_fid(mu_r, sigma_r, mu_noise, sigma_noise)
print(f"FID (random noise):     {fid_noise:.1f}")

# 2. Untrained Generator
print("\nComputing FID for untrained Generator...")
gen_untrained = Generator(z_dim).to(device)
with torch.no_grad():
    fake_untrained = []
    for _ in range(n_eval // 64 + 1):
        z = torch.randn(64, z_dim, device=device)
        fake_untrained.append(gen_untrained(z).cpu())
    fake_untrained = torch.cat(fake_untrained)[:n_eval]

untrained_feats = extract_all_features(fake_untrained, extractor)
mu_ut, sigma_ut = compute_statistics(untrained_feats)
fid_untrained = calculate_fid(mu_r, sigma_r, mu_ut, sigma_ut)
print(f"FID (untrained GAN):    {fid_untrained:.1f}")

# 3. Real vs Real (sanity check — should be near 0)
print("\nComputing FID for real vs real (sanity check)...")
# Split real images in half
half = n_eval // 2
feats_a = real_feats[:half]
feats_b = real_feats[half:]
mu_a, sigma_a = compute_statistics(feats_a)
mu_b, sigma_b = compute_statistics(feats_b)
fid_real = calculate_fid(mu_a, sigma_a, mu_b, sigma_b)
print(f"FID (real vs real):      {fid_real:.1f}  (small, not 0: the small-sample bias)")

In [ ]:
# Visualize what each looks like
fig, axes = plt.subplots(3, 8, figsize=(12, 5))

for j in range(8):
    axes[0][j].imshow((real_images[j].squeeze() + 1)/2, cmap="gray")
    axes[0][j].axis("off")
    axes[1][j].imshow((fake_untrained[j].squeeze() + 1)/2, cmap="gray")
    axes[1][j].axis("off")
    axes[2][j].imshow((random_noise[j].squeeze() + 1)/2, cmap="gray")
    axes[2][j].axis("off")

axes[0][0].set_ylabel("Real", fontsize=12, rotation=0, labelpad=40)
axes[1][0].set_ylabel("Untrained\nGAN", fontsize=11, rotation=0, labelpad=50)
axes[2][0].set_ylabel("Random\nnoise", fontsize=11, rotation=0, labelpad=50)

plt.suptitle(f"FID: Real vs Real={fid_real:.0f} | Untrained={fid_untrained:.0f} | Noise={fid_noise:.0f}",
             fontsize=13)
plt.tight_layout()
plt.show()

## 6. Experiment 2: Track FID During Training

Train a DCGAN and compute FID every few epochs.  
Watch the FID drop as the GAN learns!

**Predict:** will FID fall every time it is measured, or can it rise again?

In [ ]:
# Fresh models
gen  = Generator(z_dim).to(device)
disc = Discriminator().to(device)

opt_G = torch.optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

epochs = 20
fid_every = 4      # compute FID every N epochs
fid_history = []
fid_epochs  = []

print(f"Training DCGAN for {epochs} epochs, computing FID every {fid_every}...\n")

for epoch in range(epochs):
    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)
        real_t = torch.ones(bs, 1, device=device)
        fake_t = torch.zeros(bs, 1, device=device)

        # D step
        z = torch.randn(bs, z_dim, device=device)
        fake = gen(z).detach()
        loss_D = criterion(disc(real_imgs), real_t) + criterion(disc(fake), fake_t)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # G step
        z = torch.randn(bs, z_dim, device=device)
        fake = gen(z)
        loss_G = criterion(disc(fake), real_t)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    # Compute FID periodically
    if (epoch + 1) % fid_every == 0 or epoch == 0:
        gen.eval()
        with torch.no_grad():
            fake_eval = []
            for _ in range(n_eval // 64 + 1):
                z = torch.randn(64, z_dim, device=device)
                fake_eval.append(gen(z).cpu())
            fake_eval = torch.cat(fake_eval)[:n_eval]

        fake_feats = extract_all_features(fake_eval, extractor)
        mu_f, sigma_f = compute_statistics(fake_feats)
        fid = calculate_fid(mu_r, sigma_r, mu_f, sigma_f)

        fid_history.append(fid)
        fid_epochs.append(epoch + 1)
        print(f"Epoch {epoch+1:3d} | D: {loss_D:.4f} | G: {loss_G:.4f} | FID: {fid:.1f}")
        gen.train()
    else:
        print(f"Epoch {epoch+1:3d} | D: {loss_D:.4f} | G: {loss_G:.4f}")

In [ ]:
# Plot FID over training
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fid_epochs, fid_history, 'o-', markersize=8, linewidth=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("FID (lower = better)", fontsize=12)
ax.set_title("FID During Training", fontsize=14)
ax.grid(True, alpha=0.3)

# Annotate each point
for ep, fid in zip(fid_epochs, fid_history):
    ax.annotate(f"{fid:.0f}", (ep, fid), textcoords="offset points",
                xytext=(0, 12), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nFID dropped from {fid_history[0]:.0f} to {fid_history[-1]:.0f}")
print("Lower FID = better image quality + diversity!")

## 7. See the Improvement

Compare generated images at different FID levels.

In [ ]:
# Show final generated images
gen.eval()
with torch.no_grad():
    z = torch.randn(16, z_dim, device=device)
    samples = gen(z).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img = (samples[i].squeeze() + 1) / 2
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle(f"Generated Images (FID = {fid_history[-1]:.1f})", fontsize=14)
plt.tight_layout()
plt.show()
gen.train();

## 8. Experiment 3: Feature Space Visualization

Let's see what the Inception features look like using t-SNE.  
Real and fake features should overlap if the GAN is good.

t-SNE is a picture, not a metric: it distorts distances, so use it only as a visual sanity check.

In [ ]:
from sklearn.manifold import TSNE

# Use a subset for speed
n_vis = 500
real_sub = real_feats[:n_vis]

gen.eval()
with torch.no_grad():
    fake_vis = []
    for _ in range(n_vis // 64 + 1):
        z = torch.randn(64, z_dim, device=device)
        fake_vis.append(gen(z).cpu())
    fake_vis = torch.cat(fake_vis)[:n_vis]
gen.train()

fake_sub = extract_all_features(fake_vis, extractor)[:n_vis]

# Combine and run t-SNE
combined = np.vstack([real_sub, fake_sub])
labels_vis = ['Real'] * n_vis + ['Fake'] * n_vis

print(f"Running t-SNE on {len(combined)} feature vectors...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embedded = tsne.fit_transform(combined)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(embedded[:n_vis, 0], embedded[:n_vis, 1],
           alpha=0.4, s=10, label='Real', c='blue')
ax.scatter(embedded[n_vis:, 0], embedded[n_vis:, 1],
           alpha=0.4, s=10, label='Fake', c='red')
ax.legend(fontsize=12)
ax.set_title(f"t-SNE of Inception Features (FID = {fid_history[-1]:.1f})", fontsize=13)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

print("More overlap between blue and red = lower FID = better GAN!")

## Key Takeaways

1. **FID uses Inception pool3 features** — 2048-dim semantic representation
2. **Preprocessing matters** — resize to 299x299, 3 channels, ImageNet normalization
3. **FID = center distance + shape distance** between real and fake feature distributions
4. **Lower FID = better** — 0 is perfect, 100+ is poor
5. **Track FID during training** — it should decrease as the GAN improves
6. **Real vs Real FID is small, not 0** — small samples bias FID upward; compare models only at the same sample count
7. **t-SNE overlap** — visual confirmation that features distributions match

**Next: L14 (GAN bias and fairness)** — what happens when the training data is biased, and why one overall FID can hide it?